**PADDLEOCR Generate Real World Dataset**

**Step 1: Mount Google Drive and Install PaddlecOCR**

In [1]:
# Install PaddlePaddle and PaddleOCR (CPU version)
!pip install paddlepaddle
!pip install paddleocr
!pip install pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.0/87.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# Install compatible langchain and restart runtime
!pip install "langchain<0.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.2
    Uninstalling tenacity-9.1.2:
      Successfully uninstalled tenacity-9.1.2
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: numpy
    Found existing

**Step 2: Start Generating Dataset**

In [45]:
%cd /content
!rm -rf cropped
!rm -rf gt.txt

/content


In [ ]:
import os
import cv2
import numpy as np
from paddlex import create_model, create_pipeline

# Configuration
SAMPLE_DIR = '/content/drive/MyDrive/all_sample_images/eval_images/'  # Folder with sample images (train and/or eval)
CROPPED_DIR = '/content/cropped/'  # Output folder for cropped images
GT_FILE = '/content/gt.txt'  # Ground truth file

# Create output directory
os.makedirs(CROPPED_DIR, exist_ok=True)

# Supported image extensions
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# List all sample images
sample_files = [f for f in os.listdir(SAMPLE_DIR) if os.path.splitext(f.lower())[1] in IMG_EXTS]

# Load detection and recognition models
DET_MODEL_NAME = 'PP-OCRv5_server_det'
REC_MODEL_NAME = 'PP-OCRv5_server_rec'
det_model = create_model(model_name=DET_MODEL_NAME)  # Assuming auto-download
rec_model = create_model(model_name=REC_MODEL_NAME)

# Open ground truth file for writing
with open(GT_FILE, 'w', encoding='utf-8') as gt_f:
    total_crops = 0
    for img_file in sample_files:
        img_path = os.path.join(SAMPLE_DIR, img_file)
        print(f"Processing {img_file}...")

        # 1. Run Detection and CONVERT TO LIST
        det_results_generator = det_model.predict(img_path)
        det_results_list = list(det_results_generator)
        
        if det_results_list:
            det_output = det_results_list[0]
        else:
            print(f"  No detection results for {img_file}")
            continue

        # 2. Load the image for cropping
        img = cv2.imread(img_path)
        if img is None:
            print(f"  Failed to load {img_file}")
            continue

        # 3. Process the Detection polygons and crop
        # The 'dt_polys' key contains a list of all detected polygon arrays.
        all_polygons = det_output.get('dt_polys', [])

        cropped_images = []
        for polygon in all_polygons:
            # Reshape the polygon array into a list of [x, y] coordinates
            polygon_points = np.array(polygon).reshape(-1, 2)

            # Calculate the Minimum Bounding Rectangle (MBR)
            # Find the min/max x and y across all points in the polygon
            xmin = np.min(polygon_points[:, 0]).astype(int)
            ymin = np.min(polygon_points[:, 1]).astype(int)
            xmax = np.max(polygon_points[:, 0]).astype(int)
            ymax = np.max(polygon_points[:, 1]).astype(int)

            # Ensure coordinates are within image boundaries
            xmin, ymin = max(0, xmin), max(0, ymin)
            xmax, ymax = min(img.shape[1], xmax), min(img.shape[0], ymax)

            # Crop the image using slicing
            crop = img[ymin:ymax, xmin:xmax]

            # Optional: Filter out tiny crops that might cause model errors
            if crop.shape[0] > 5 and crop.shape[1] > 5:
                cropped_images.append(crop)

        # --- Running Recognition on Cropped Images ---
        print(f"--- Running Recognition on {len(cropped_images)} Cropped Images ---")

        for i, crop in enumerate(cropped_images):
            # Ensure the crop is valid 
            if crop.size == 0:
                continue

            # The custom recognition model predicts the text and score for the single crop
            rec_results_generator = rec_model.predict(crop)

            # Convert the generator to a list
            rec_results = list(rec_results_generator)

            # PaddleX recognition prediction returns a list containing one dictionary
            if rec_results and isinstance(rec_results[0], dict):
                text = rec_results[0].get('rec_text', 'N/A')
                score = rec_results[0].get('rec_score', 0.0)
            else:
                # Handle unexpected output format
                print(f"Warning: Recognition failed for crop {i+1}. Result format was unexpected.")
                continue

            # Save cropped image
            base_name = os.path.splitext(img_file)[0]
            cropped_name = f"{base_name}_text_{i}.jpg"
            cropped_path = os.path.join(CROPPED_DIR, cropped_name)
            cv2.imwrite(cropped_path, crop)

            # Write to ground truth file
            gt_line = f"{cropped_name}\t{text}\n"
            gt_f.write(gt_line)
            gt_f.flush()

            print(f"  Crop {i+1}: Text '{text}' (score: {score:.2f}) | Saved to {cropped_path}")
            total_crops += 1

print(f"Dataset generated! Total cropped images: {total_crops}")
print(f"- Cropped images: {CROPPED_DIR}")
print(f"- Ground truth: {GT_FILE}")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_rec`.


Processing 20251024_225854.jpg...
--- Running Recognition on 33 Cropped Images ---
  Crop 1: Text '多謝惠顧，塑膠購物袋收費想不設退款。' (score: 0.88) | Saved to /content/cropped/20251024_225854_text_0.jpg
  Crop 2: Text '上一次於2025-10-14現金增值' (score: 0.87) | Saved to /content/cropped/20251024_225854_text_1.jpg
  Crop 3: Text '317.00' (score: 1.00) | Saved to /content/cropped/20251024_225854_text_2.jpg
  Crop 4: Text '八達通餘額' (score: 0.93) | Saved to /content/cropped/20251024_225854_text_3.jpg


**Step 3: Copy Output Result**

In [43]:
!cp -r /content/cropped /content/drive/MyDrive/
!cp -r /content/gt.txt /content/drive/MyDrive/cropped